In [30]:
import boto3
from botocore.client import Config

s3 = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

## Bucket Existente

In [31]:
BUCKET = "meu-novo-bucket"

existentes = [b['Name'] for b in s3.list_buckets()['Buckets']]
if BUCKET not in existentes:
    s3.create_bucket(Bucket=BUCKET)
print([b['Name'] for b in s3.list_buckets()['Buckets']])

['meu-bucket', 'meu-novo-bucket']


### PUT

In [32]:
with open('localfile.txt', 'w') as f:
    f.write('Olá, MinIO! Primeiro objeto do data lake.\n')


s3.upload_file('localfile.txt', 'meu-bucket', 'localfile.txt')

### GET

In [33]:
s3.download_file('meu-bucket', 'localfile.txt', 'baixado_locafile.txt')

### Listando objeto em um bucket

In [34]:
response = s3.list_objects_v2(Bucket='meu-bucket')


for obj in response.get('Contents', []):
   print(obj['Key'])

localfile.txt


### Deletando um objeto

In [35]:
s3.delete_object(
    Bucket='meu-bucket',
    Key='localfile.txt'
)

{'ResponseMetadata': {'RequestId': '18E08B7C5C690ECF',
  'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
  'HTTPStatusCode': 204,
  'HTTPHeaders': {'accept-ranges': 'bytes',
   'server': 'MinIO',
   'strict-transport-security': 'max-age=31536000; includeSubDomains',
   'vary': 'Origin, Accept-Encoding',
   'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
   'x-amz-request-id': '18E08B7C5C690ECF',
   'x-content-type-options': 'nosniff',
   'x-ratelimit-limit': '351',
   'x-ratelimit-remaining': '351',
   'x-xss-protection': '1; mode=block',
   'date': 'Wed, 21 Oct 2026 12:33:37 GMT'},
  'RetryAttempts': 0}}

### Presigned URL

In [36]:
url = s3.generate_presigned_url(
    'get_object',
    Params={
        'Bucket': 'meu-novo-bucket',
        'Key': 'localfile.txt'
    },
    ExpiresIn=3600
)

In [40]:
endpoint = s3.meta.endpoint_url

import urllib.request, urllib.error

try:
    urllib.request.urlopen(f'{endpoint}/{BUCKET}/localfile.txt')
except urllib.error.HTTPError as e:
    print('Sem assinatura:', e.code, e.reason)

print('Com presigned URL:', urllib.request.urlopen(url).read().decode())

Sem assinatura: 403 Forbidden
Com presigned URL: Ola, MinIO! Teste via curl.

